# grok-010 · GSM8K QLoRA SFT + 同尺对比（学习向）

> **对应路线 1+2**：把 008 的 QLoRA 工程接到 009 的硬尺子上。  
> **唯一主指标**：GSM8K 子集 exact match（与 009 同 seed=42、同 N）。
>
> **流程**
> 1. 重建/对齐 `gsm8k_eval_v1`（同 seed）  
> 2. 用 **train 切分** 做 QLoRA SFT（绝不碰 test 子集 id）  
> 3. 对比 **base vs sft** 的 EM  
>
> **模型**：默认 `Qwen/Qwen2.5-1.5B-Instruct` + 4bit QLoRA（配额友好；可改 3B）


In [ ]:
# 【步骤】训练可见双卡；生成评测时用主卡
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


In [ ]:
# 【步骤】环境
import os, re, json, time, random, math, gc, platform
from pathlib import Path
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

OUT = Path("/kaggle/working")
EVAL = OUT / "gsm8k_eval_v1"
OUT.mkdir(parents=True, exist_ok=True)
EVAL.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available()
print("gpus", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DEVICE = torch.device("cuda:0")
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_EVAL = 100
MAX_TRAIN = 800  # 学习向：小步验证链路；可加大
STEPS = 200


In [ ]:
# 【步骤】数据：train 训练；test 固定子集只评测
from datasets import load_dataset

test = load_dataset("openai/gsm8k", "main", split="test")
train = load_dataset("openai/gsm8k", "main", split="train")

def parse_gold(answer: str) -> str:
    m = re.search(r"####\s*([-0-9.,]+)", answer)
    if m:
        return m.group(1).replace(",", "")
    return answer.strip().split()[-1]

idx = list(range(len(test)))
random.Random(SEED).shuffle(idx)
eval_idx = set(idx[:N_EVAL])
eval_rows = []
for i in idx[:N_EVAL]:
    ex = test[int(i)]
    eval_rows.append({
        "id": f"gsm8k_{i}",
        "question": ex["question"],
        "gold": parse_gold(ex["answer"]),
    })

# 训练样本来自 train split（天然不与 test id 相交）
train_rows = []
for i, ex in enumerate(train):
    if len(train_rows) >= MAX_TRAIN:
        break
    train_rows.append({
        "question": ex["question"],
        "gold": parse_gold(ex["answer"]),
        "full": ex["answer"],
    })
print("train", len(train_rows), "eval", len(eval_rows))
(EVAL/"manifest.json").write_text(json.dumps({
    "seed": SEED, "n_eval": len(eval_rows), "n_train_used": len(train_rows), "model_id": MODEL_ID,
}, indent=2))


In [ ]:
# 【步骤】QLoRA 加载 + LoRA 注入
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from torch.utils.data import Dataset, DataLoader

try:
    import peft.tuners.lora.torchao as peft_torchao
    peft_torchao.is_torchao_available = lambda: False
except Exception:
    pass

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map={"": 0}, trust_remote_code=True,
)
base = prepare_model_for_kbit_training(base)
base.gradient_checkpointing_enable()
if hasattr(base, "enable_input_require_grads"):
    base.enable_input_require_grads()

cand = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
present = sorted({n.split(".")[-1] for n,_ in base.named_modules() if n.split(".")[-1] in cand})
model = get_peft_model(base, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", target_modules=present,
))
model.print_trainable_parameters()


In [ ]:
# 【步骤】SFT 数据格式：问题 + 完整解题文本（含 #### 答案）
class GSMSFT(Dataset):
    def __init__(self, rows, max_len=768):
        self.feats = []
        for r in rows:
            user = r["question"] + "\n\nSolve step by step. End with #### <number>."
            # full 字段含推理与 #### gold
            assistant = r.get("full") or f"#### {r['gold']}"
            text = tokenizer.apply_chat_template(
                [{"role":"user","content":user},{"role":"assistant","content":assistant}],
                tokenize=False, add_generation_prompt=False,
            )
            self.feats.append(tokenizer(text, truncation=True, max_length=max_len, padding=False))
    def __len__(self):
        return len(self.feats)
    def __getitem__(self, i):
        return self.feats[i]

def collate(batch):
    return tokenizer.pad(batch, padding=True, return_tensors="pt")

loader = DataLoader(GSMSFT(train_rows), batch_size=1, shuffle=True, collate_fn=collate)
print("batches", len(loader))


In [ ]:
# 【步骤】QLoRA 短训
from torch.optim import AdamW

opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
model.train()
losses = []
it = iter(loader)
t0 = time.perf_counter()
for step in range(STEPS):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader); batch = next(it)
    batch = {k: v.to(DEVICE) for k,v in batch.items()}
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    opt.zero_grad(set_to_none=True)
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=labels)
    loss = out.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    opt.step()
    losses.append(float(loss.detach().float().cpu()))
    if step % 20 == 0 or step == STEPS-1:
        print(f"step {step}/{STEPS} loss={losses[-1]:.4f}")
train_s = time.perf_counter() - t0
print("train_s", train_s)
adapter = OUT / "adapter_gsm8k_qlora"
adapter.mkdir(exist_ok=True)
model.save_pretrained(adapter)
tokenizer.save_pretrained(adapter)


In [ ]:
# 【步骤】同尺评测：base（关 adapter）vs sft
def extract_final_number(text: str):
    if not text:
        return None
    m = re.search(r"####\s*([-0-9.,]+)", text)
    if m:
        return m.group(1).replace(",", "")
    # boxed
    m = re.search(r"boxed\{([^}]+)\}", text)
    if m:
        nums = re.findall(r"-?\d+(?:\.\d+)?", m.group(1).replace(",", ""))
        if nums:
            return nums[-1]
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None

@torch.no_grad()
def generate_with(m, question, max_new=256):
    m.eval()
    prompt = (
        question.strip()
        + "\n\nWrite a step by step solution. Put the final answer after the steps as #### <number>."
    )
    messages = [{"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    out = m.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def eval_em(m, tag):
    sc = []
    for i, r in enumerate(eval_rows):
        pred = generate_with(m, r["question"])
        pn, gn = extract_final_number(pred), extract_final_number(r["gold"])
        ok = 0.0
        if pn is not None and gn is not None:
            try:
                ok = float(float(pn) == float(gn))
            except Exception:
                ok = float(pn == gn)
        sc.append(ok)
        if (i+1) % 20 == 0:
            print(f"{tag} {i+1}/{len(eval_rows)} em={sum(sc)/len(sc):.3f}")
    em = sum(sc)/len(sc)
    print(tag, "EM", em)
    return em

# base：临时 disable adapter
with model.disable_adapter():
    base_em = eval_em(model, "base")
sft_em = eval_em(model, "sft")
rel = (sft_em - base_em) / max(1e-8, base_em) if base_em > 0 else None
print("relative_gain", rel)


In [ ]:
# 【步骤】报告
report = {
    "notebook": "grok-010-gsm8k-qlora-sft",
    "phase": "deep_D_closed_loop",
    "model_id": MODEL_ID,
    "n_train": len(train_rows),
    "n_eval": len(eval_rows),
    "steps": STEPS,
    "loss_start": losses[0],
    "loss_end": losses[-1],
    "train_seconds": train_s,
    "base_em": base_em,
    "sft_em": sft_em,
    "abs_gain": sft_em - base_em,
    "relative_gain": rel,
    "adapter": str(adapter),
    "next": "grok-011-single-vs-dual-t4 测吞吐；或加强数据/步数再冲 EM",
}
(OUT/"grok010_results.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(json.dumps(report, indent=2, ensure_ascii=False)[:2000])
print("DONE grok-010")


## 学习检查清单
- train 为何必须来自 GSM8K **train** split？  
- base EM 与 sft EM 如何同 decode、同子集比较？  
- 若 sft 不涨：是步数不够、数据不够，还是 1.5B 能力天花板？  
